In [1]:
import tensorflow as tf
from src.dataset import build_yolo_dataset
import numpy as np
from src.constants import ANCHORS, GRID_SIZE, IMAGE_SIZE
from src.model import build_model

In [2]:
val_dataset = build_yolo_dataset(batch_size= 16, split= "train", shuffle= False)

In [3]:
for batch in val_dataset.take(1):
    images = batch["image"]
    true_boxes = batch["boxes"]
    true_labels = batch["labels"]
    targets = batch["targets"]
print("Images shape:", images.shape)
print("Targets shape:", targets.shape)

Images shape: (16, 224, 224, 3)
Targets shape: (16, 14, 14, 3, 25)


In [4]:
model = build_model()
checkpoint = tf.train.Checkpoint(model=model)
checkpoint.restore(
    "new_checkpoints/ckpt-31"
).expect_partial()
preds = model(images, training=False)
print("Predictions shape:", preds.shape)

Predictions shape: (16, 14, 14, 3, 25)


In [22]:
image = batch["image"][1]
pred = preds[1]

grid_h, grid_w = GRID_SIZE

detections = []

for cell_y in range(grid_h):
    for cell_x in range(grid_w):
        for anchor_idx in range(len(ANCHORS)):
            raw_objectness = pred[cell_y, cell_x, anchor_idx, 0]
            tx = pred[cell_y, cell_x, anchor_idx, 1]
            ty = pred[cell_y, cell_x, anchor_idx, 2]
            tw = pred[cell_y, cell_x, anchor_idx, 3]
            th = pred[cell_y, cell_x, anchor_idx, 4]
            raw_class_logits = pred[cell_y, cell_x, anchor_idx, 5:]

            # Decode center
            cx = (tx + cell_x) / grid_w
            cy = (ty + cell_y) / grid_h

            # Decode width/height
            anchor_w, anchor_h = ANCHORS[anchor_idx]

            bw = np.exp(tw) * anchor_w
            bh = np.exp(th) * anchor_h

            # Convert to corners
            xmin = cx - bw / 2
            ymin = cy - bh / 2
            xmax = cx + bw / 2
            ymax = cy + bh / 2

            # Convert normalized coordinates to pixels
            xmin *= IMAGE_SIZE[1]
            ymin *= IMAGE_SIZE[0]
            xmax *= IMAGE_SIZE[1]
            ymax *= IMAGE_SIZE[0]

            objectness = tf.sigmoid(raw_objectness)
            class_probs = tf.nn.softmax(raw_class_logits)

            class_id = np.argmax(class_probs)
            class_prob = class_probs[class_id]

            confidence = objectness * class_prob

            detections.append({
               "box": [
               float(xmin),
               float(ymin),
               float(xmax),
               float(ymax)
                 ],
              "objectness": float(objectness),
              "class_id": int(class_id),
              "class_prob": float(class_prob),
              "confidence": float(confidence),
             })

In [28]:
CONF_THRESHOLD = 0.04

filtered_detections = [
    d for d in detections
    if d["confidence"] >= CONF_THRESHOLD
]

print("Total predictions:", len(detections))
print("After confidence filtering:", len(filtered_detections))

for d in filtered_detections:
    d["box"] = [
        max(0, min(224, x))
        for x in d["box"]
    ]

Total predictions: 588
After confidence filtering: 2


In [29]:
for d in sorted(
    detections,
    key=lambda x: x["confidence"],
    reverse=True
):
    if d["confidence"] >= 0.04:
        print(
            f"class={d['class_id']}, "
            f"confidence={d['confidence']:.3f}, "
            f"objectness={d['objectness']:.3f}, "
            f"class_prob={d['class_prob']:.3f}, "
            f"box={d['box']}"
        )

class=13, confidence=0.047, objectness=0.223, class_prob=0.209, box=[12.017878532409668, 15.272418975830078, 217.47776794433594, 216.0323486328125]
class=13, confidence=0.043, objectness=0.157, class_prob=0.274, box=[0.1702108383178711, 14.73721981048584, 208.70736694335938, 216.84796142578125]


In [31]:
print("Ground truth:")

for box, label in zip(
    true_boxes[1].numpy(),
    true_labels[1].numpy()
):
    print(
        f"class={int(label)}, "
        f"box={box}"
    )

Ground truth:
class=4, box=[120.96001      0.68501526 169.34401    120.56268   ]
class=4, box=[ 25.536001     0.68501526  73.472      102.75229   ]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]
class=0, box=[0. 0. 0. 0.]


In [32]:
target = targets[1]

positive_indices = tf.where(target[..., 0] == 1)

print("Positive locations:")
print(positive_indices.numpy())

Positive locations:
[[3 3 1]
 [3 9 1]]


In [20]:
for cell_y, cell_x, anchor_idx in positive_indices.numpy():

    target_values = target[cell_y, cell_x, anchor_idx]

    true_class = int(tf.argmax(target_values[5:]))

    print(
        f"Cell=({cell_x}, {cell_y}), "
        f"Anchor={anchor_idx}"
    )

    print("  objectness:", float(target_values[0]))
    print("  tx:", float(target_values[1]))
    print("  ty:", float(target_values[2]))
    print("  tw:", float(target_values[3]))
    print("  th:", float(target_values[4]))
    print("  class:", true_class)

Cell=(6, 3), Anchor=1
  objectness: 1.0
  tx: 0.25800037384033203
  ty: 0.3416292667388916
  tw: -0.49217963218688965
  th: -0.6714216470718384
  class: 14
Cell=(7, 8), Anchor=2
  objectness: 1.0
  tx: 0.3360004425048828
  ty: 0.029412269592285156
  tw: -0.4442563056945801
  th: -0.4736223816871643
  class: 12


In [33]:
for cell_y, cell_x, anchor_idx in positive_indices.numpy():

    target_values = target[cell_y, cell_x, anchor_idx]
    pred_values = pred[cell_y, cell_x, anchor_idx]

    pred_objectness = float(tf.sigmoid(pred_values[0]))

    pred_class_probs = tf.nn.softmax(pred_values[5:])
    pred_class = int(tf.argmax(pred_class_probs))
    pred_class_prob = float(pred_class_probs[pred_class])

    true_class = int(tf.argmax(target_values[5:]))

    print(f"\nCell=(x={cell_x}, y={cell_y}), Anchor={anchor_idx}")
    print(f"True class: {true_class}")
    print(f"Pred class: {pred_class}")
    print(f"Pred class probability: {pred_class_prob:.3f}")

    print(f"\nObjectness:")
    print(f"  target:     {float(target_values[0]):.3f}")
    print(f"  prediction: {pred_objectness:.3f}")

    print("\nBox:")
    print(f"  tx: target={float(target_values[1]):.3f}, "
          f"pred={float(pred_values[1]):.3f}")
    print(f"  ty: target={float(target_values[2]):.3f}, "
          f"pred={float(pred_values[2]):.3f}")
    print(f"  tw: target={float(target_values[3]):.3f}, "
          f"pred={float(pred_values[3]):.3f}")
    print(f"  th: target={float(target_values[4]):.3f}, "
          f"pred={float(pred_values[4]):.3f}")


Cell=(x=3, y=3), Anchor=1
True class: 4
Pred class: 14
Pred class probability: 0.415

Objectness:
  target:     1.000
  prediction: 0.009

Box:
  tx: target=0.094, pred=0.432
  ty: target=0.232, pred=0.531
  tw: target=-0.695, pred=-0.878
  th: target=-0.178, pred=-0.414

Cell=(x=9, y=3), Anchor=1
True class: 4
Pred class: 19
Pred class probability: 0.335

Objectness:
  target:     1.000
  prediction: 0.007

Box:
  tx: target=0.072, pred=0.510
  ty: target=0.789, pred=0.542
  tw: target=-0.685, pred=-0.594
  th: target=-0.017, pred=-0.441


In [33]:
from src.loss import YOLOLoss
model_test = build_model()

loss_fn_test = YOLOLoss(
    lambda_box=5.0,
    lambda_obj=1.0,
    lambda_noobj=0.5,
    lambda_class=1.0
)

In [ ]:
images_test = images[:1]
targets_test = targets[:1]

with tf.GradientTape() as tape:
    predictions_test = model_test(images_test, training=True)

    loss, box_loss, obj_loss, no_obj_loss, class_loss = \
        loss_fn_test.compute_losses_components(
            targets_test,
            predictions_test
        )

gradients = tape.gradient(loss, model_test.trainable_variables)

print("Loss:", float(loss))
print("Box:", float(box_loss))
print("Obj:", float(obj_loss))
print("NoObj:", float(no_obj_loss))
print("Class:", float(class_loss))

print("\nGradient check:")
for variable, gradient in zip(model_test.trainable_variables, gradients):
    print(
        variable.name,
        "gradient is None:", gradient is None
    )

Loss: 116.04615783691406
Box: 11.16722297668457
Obj: 0.2837398052215576
NoObj: 112.41119384765625
Class: 3.7207069396972656

Gradient check:
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
gamma gradient is None: False
beta gradient is None: False
kernel gradient is None: False
kernel gradient is

In [101]:
single_image = images[:8]
single_target = targets[:8]


optimizer_test = tf.keras.optimizers.Adam(learning_rate=1e-3)

for step in range(1000):

    with tf.GradientTape() as tape:
        prediction = model_test(single_image, training=True)

        total_loss, box_loss, obj_loss, no_obj_loss, class_loss = \
            loss_fn_test.compute_losses_components(
                single_target,
                prediction
            )

    gradients = tape.gradient(
        total_loss,
        model_test.trainable_variables
    )

    optimizer_test.apply_gradients(
        zip(gradients, model_test.trainable_variables)
    )

    if step % 100 == 0:
        print(
            step,
            float(total_loss),
            float(box_loss),
            float(obj_loss),
            float(no_obj_loss),
            float(class_loss)
        )

0 210.6611785888672 38.00579833984375 10.688793182373047 2.075448989868164 8.90566349029541
100 0.8076590895652771 0.052879698574543 0.23726961016654968 0.5337925553321838 0.03909476846456528
200 0.21190336346626282 0.02269606664776802 0.034000858664512634 0.0946338027715683 0.017105277627706528
300 0.11257230490446091 0.01596427522599697 0.012388980947434902 0.025258848443627357 0.007732517085969448
400 0.06709884107112885 0.008781610056757927 0.006739849224686623 0.025080841034650803 0.0039105224423110485
500 0.044607069343328476 0.006686125881969929 0.003362247720360756 0.010523710399866104 0.0025523416697978973
600 0.07773401588201523 0.013612071983516216 0.0038158586248755455 0.007651303894817829 0.002032143995165825
700 0.03129681199789047 0.004443192854523659 0.002859721891582012 0.007638562470674515 0.0024018443655222654
800 0.026287337765097618 0.0039792973548173904 0.0018161400221288204 0.005596713162958622 0.0017763536889106035
900 0.014699007384479046 0.001990857068449259 0

In [102]:
preds = model_test(single_image, training=False)

In [31]:
print("Total predictions:", len(detections))

confidences = np.array(
    [d["confidence"] for d in detections]
)

print("Confidence range:",
      confidences.min(),
      confidences.max())

print("Top 20 confidences:")
print(np.sort(confidences)[-20:][::-1])

Total predictions: 588
Confidence range: 5.674232670571655e-05 0.30357053875923157
Top 20 confidences:
[0.30357054 0.29832482 0.28176636 0.26599911 0.25021592 0.24520886
 0.24451573 0.24281515 0.238381   0.22514142 0.22262116 0.22206491
 0.22088812 0.22004239 0.2196603  0.21774812 0.21329416 0.2104712
 0.20666748 0.20465362]


In [15]:
best = sorted(
    detections,
    key=lambda d: d["confidence"],
    reverse=True
)[:10]

for d in best:
    print(
        f"class={d['class_id']}, "
        f"confidence={d['confidence']:.3f}, "
        f"objectness={d['objectness']:.3f}, "
        f"class_prob={d['class_prob']:.3f}, "
        f"box={d['box']}"
    )

class=7, confidence=0.037, objectness=0.144, class_prob=0.256, box=[11.06308364868164, 20.79825210571289, 218.8167266845703, 216.5279998779297]
class=7, confidence=0.033, objectness=0.120, class_prob=0.278, box=[25.50952911376953, 40.938507080078125, 182.2789306640625, 164.29983520507812]
class=7, confidence=0.033, objectness=0.103, class_prob=0.322, box=[33.824073791503906, 46.76396942138672, 197.6829071044922, 161.05654907226562]
class=2, confidence=0.027, objectness=0.092, class_prob=0.293, box=[48.25257110595703, 69.94881439208984, 157.48992919921875, 166.83847045898438]
class=7, confidence=0.024, objectness=0.086, class_prob=0.279, box=[21.413593292236328, 36.58377456665039, 188.635498046875, 206.13150024414062]
class=17, confidence=0.023, objectness=0.046, class_prob=0.488, box=[6.157297134399414, 54.512081146240234, 218.4456787109375, 225.14608764648438]
class=2, confidence=0.014, objectness=0.035, class_prob=0.415, box=[49.34545135498047, 88.1598129272461, 157.98074340820312, 1

In [16]:
objectnesses = np.array(
    [d["objectness"] for d in detections]
)

class_probs = np.array(
    [d["class_prob"] for d in detections]
)

print(
    "Objectness range:",
    objectnesses.min(),
    objectnesses.max()
)

print(
    "Best class probability range:",
    class_probs.min(),
    class_probs.max()
)

Objectness range: 5.823486224493246e-13 0.14419838786125183
Best class probability range: 0.1289171427488327 1.0


In [34]:
target = targets[1]
pred = preds[1]

positive_indices = tf.where(target[..., 0] == 1)

for cell_y, cell_x, anchor_idx in positive_indices.numpy():

    target_values = target[cell_y, cell_x, anchor_idx]
    pred_values = pred[cell_y, cell_x, anchor_idx]

    objectness = float(tf.sigmoid(pred_values[0]))

    class_probs = tf.nn.softmax(pred_values[5:])
    pred_class = int(tf.argmax(class_probs))
    pred_class_prob = float(class_probs[pred_class])

    true_class = int(tf.argmax(target_values[5:]))

    print(f"\nCell=(x={cell_x}, y={cell_y}), Anchor={anchor_idx}")
    print(f"True class: {true_class}")
    print(
        f"Pred class: {pred_class}, "
        f"probability: {pred_class_prob:.3f}"
    )
    print(
        f"Objectness: target={float(target_values[0]):.3f}, "
        f"prediction={objectness:.3f}"
    )

    print(
        f"tx: target={float(target_values[1]):.3f}, "
        f"pred={float(pred_values[1]):.3f}"
    )
    print(
        f"ty: target={float(target_values[2]):.3f}, "
        f"pred={float(pred_values[2]):.3f}"
    )
    print(
        f"tw: target={float(target_values[3]):.3f}, "
        f"pred={float(pred_values[3]):.3f}"
    )
    print(
        f"th: target={float(target_values[4]):.3f}, "
        f"pred={float(pred_values[4]):.3f}"
    )


Cell=(x=3, y=3), Anchor=1
True class: 4
Pred class: 14, probability: 0.415
Objectness: target=1.000, prediction=0.009
tx: target=0.094, pred=0.432
ty: target=0.232, pred=0.531
tw: target=-0.695, pred=-0.878
th: target=-0.178, pred=-0.414

Cell=(x=9, y=3), Anchor=1
True class: 4
Pred class: 19, probability: 0.335
Objectness: target=1.000, prediction=0.007
tx: target=0.072, pred=0.510
ty: target=0.789, pred=0.542
tw: target=-0.685, pred=-0.594
th: target=-0.017, pred=-0.441


In [35]:
target = targets[0]  # or targets[0]

object_mask = target[..., 0] > 0

num_positive = int(tf.reduce_sum(tf.cast(object_mask, tf.int32)))
num_negative = int(
    tf.reduce_sum(tf.cast(~object_mask, tf.int32))
)

print("Positive slots:", num_positive)
print("Negative slots:", num_negative)
print("Total slots:", num_positive + num_negative)

Positive slots: 2
Negative slots: 586
Total slots: 588


In [36]:
true_obj = target[..., 0]
pred_obj = pred[..., 0]

object_mask = tf.cast(true_obj > 0, tf.float32)
no_object_mask = 1.0 - object_mask

bce = tf.nn.sigmoid_cross_entropy_with_logits(
    labels=true_obj,
    logits=pred_obj
)

positive_loss = tf.reduce_sum(
    bce * object_mask
)

negative_loss = tf.reduce_sum(
    bce * no_object_mask
)

print("Positive objectness loss:", float(positive_loss))
print("Negative objectness loss:", float(negative_loss))
print("Weighted negative loss:", float(0.5 * negative_loss))

Positive objectness loss: 7.7692461013793945
Negative objectness loss: 6.000152587890625
Weighted negative loss: 3.0000762939453125


In [38]:
batch_targets = targets
batch_preds = preds

true_obj = batch_targets[..., 0]
pred_obj = batch_preds[..., 0]

object_mask = tf.cast(true_obj > 0, tf.float32)
no_object_mask = 1.0 - object_mask

bce = tf.nn.sigmoid_cross_entropy_with_logits(
    labels=true_obj,
    logits=pred_obj
)

positive_loss = tf.reduce_sum(bce * object_mask)
negative_loss = tf.reduce_sum(bce * no_object_mask)

positive_count = tf.reduce_sum(object_mask)
negative_count = tf.reduce_sum(no_object_mask)

print("Positive slots:", int(positive_count))
print("Negative slots:", int(negative_count))

print("Positive BCE:", float(positive_loss))
print("Negative BCE:", float(negative_loss))
print("Weighted negative BCE:", float(0.5 * negative_loss))

print("Mean positive BCE:",
      float(positive_loss / tf.maximum(positive_count, 1.0)))

print("Mean negative BCE:",
      float(negative_loss / tf.maximum(negative_count, 1.0)))

Positive slots: 42
Negative slots: 9366
Positive BCE: 129.05062866210938
Negative BCE: 92.3569107055664
Weighted negative BCE: 46.1784553527832
Mean positive BCE: 3.072633981704712
Mean negative BCE: 0.00986087042838335


In [39]:
bn_layers = [
    layer for layer in model.layers
    if isinstance(layer, tf.keras.layers.BatchNormalization)
]

print("Number of BatchNorm layers:", len(bn_layers))

Number of BatchNorm layers: 22


In [40]:
for i, layer in enumerate(bn_layers[:5]):
    print(f"\nBN layer {i}: {layer.name}")
    print("moving_mean shape:", layer.moving_mean.shape)
    print("moving_variance shape:", layer.moving_variance.shape)
    print("gamma shape:", layer.gamma.shape)
    print("beta shape:", layer.beta.shape)


BN layer 0: batch_normalization
moving_mean shape: (32,)
moving_variance shape: (32,)
gamma shape: (32,)
beta shape: (32,)

BN layer 1: batch_normalization_1
moving_mean shape: (32,)
moving_variance shape: (32,)
gamma shape: (32,)
beta shape: (32,)

BN layer 2: batch_normalization_2
moving_mean shape: (32,)
moving_variance shape: (32,)
gamma shape: (32,)
beta shape: (32,)

BN layer 3: batch_normalization_4
moving_mean shape: (64,)
moving_variance shape: (64,)
gamma shape: (64,)
beta shape: (64,)

BN layer 4: batch_normalization_5
moving_mean shape: (64,)
moving_variance shape: (64,)
gamma shape: (64,)
beta shape: (64,)


In [41]:
for i, layer in enumerate(bn_layers):
    mean_ok = tf.reduce_all(tf.math.is_finite(layer.moving_mean))
    var_ok = tf.reduce_all(tf.math.is_finite(layer.moving_variance))

    print(
        f"{i:02d} {layer.name}: "
        f"mean finite={bool(mean_ok)}, "
        f"variance finite={bool(var_ok)}, "
        f"variance min={float(tf.reduce_min(layer.moving_variance)):.6f}, "
        f"variance max={float(tf.reduce_max(layer.moving_variance)):.6f}"
    )

00 batch_normalization: mean finite=True, variance finite=True, variance min=0.003496, variance max=0.143078
01 batch_normalization_1: mean finite=True, variance finite=True, variance min=1.636181, variance max=30.313459
02 batch_normalization_2: mean finite=True, variance finite=True, variance min=1.532608, variance max=14.781713
03 batch_normalization_4: mean finite=True, variance finite=True, variance min=1.510428, variance max=27.314154
04 batch_normalization_5: mean finite=True, variance finite=True, variance min=1.702561, variance max=57.486099
05 batch_normalization_3: mean finite=True, variance finite=True, variance min=0.095049, variance max=3.213977
06 batch_normalization_6: mean finite=True, variance finite=True, variance min=4.485761, variance max=80.422539
07 batch_normalization_7: mean finite=True, variance finite=True, variance min=2.474661, variance max=68.210899
08 batch_normalization_9: mean finite=True, variance finite=True, variance min=4.099209, variance max=174.73

In [42]:
image_test = images[1:2]
pred_inference = model(image_test, training=False)
pred_training = model(image_test, training=True)
obj_inference = pred_inference[..., 0]
obj_training = pred_training[..., 0]

print(
    "Objectness logits, training=False:",
    float(tf.reduce_min(obj_inference)),
    float(tf.reduce_max(obj_inference))
)

print(
    "Objectness logits, training=True:",
    float(tf.reduce_min(obj_training)),
    float(tf.reduce_max(obj_training))
)

Objectness logits, training=False: -16.928730010986328 -1.246725082397461
Objectness logits, training=True: -23.56913185119629 -0.5587449073791504


In [43]:
target = targets[1]
positive_indices = tf.where(target[..., 0] == 1)

for cell_y, cell_x, anchor_idx in positive_indices.numpy():

    logit_false = pred_inference[
        0, cell_y, cell_x, anchor_idx, 0
    ]

    logit_true = pred_training[
        0, cell_y, cell_x, anchor_idx, 0
    ]

    print(
        f"Cell=({cell_x},{cell_y}), anchor={anchor_idx}"
    )
    print(
        "  objectness training=False:",
        float(tf.sigmoid(logit_false))
    )
    print(
        "  objectness training=True:",
        float(tf.sigmoid(logit_true))
    )

Cell=(3,3), anchor=1
  objectness training=False: 0.009350200183689594
  objectness training=True: 0.0017943981802091002
Cell=(9,3), anchor=1
  objectness training=False: 0.007009080145508051
  objectness training=True: 0.01506783626973629


In [44]:
from collections import Counter
from src.dataset import train_ids, parse_annotation
from src.dataset import ID_TO_CLASS

class_counts = Counter()

for img_id in train_ids:
    boxes, labels = parse_annotation(img_id)

    for label in labels:
        class_counts[int(label)] += 1

print("Objects per class:\n")

for class_id in range(len(ID_TO_CLASS)):
    class_name = ID_TO_CLASS[class_id]
    count = class_counts[class_id]

    print(
        f"{class_id:2d}  {class_name:12s}  {count:5d}"
    )

Objects per class:

 0  aeroplane       470
 1  bicycle         410
 2  bird            592
 3  boat            508
 4  bottle          749
 5  bus             317
 6  car            1191
 7  cat             609
 8  chair          1457
 9  cow             355
10  diningtable     373
11  dog             768
12  horse           377
13  motorbike       375
14  person         5019
15  pottedplant     557
16  sheep           509
17  sofa            399
18  train           327
19  tvmonitor       412


In [45]:
counts = [
    class_counts[i]
    for i in range(len(ID_TO_CLASS))
]

print("\nMost frequent class:",
      ID_TO_CLASS[counts.index(max(counts))],
      max(counts))

print("Least frequent class:",
      ID_TO_CLASS[counts.index(min(counts))],
      min(counts))

print(
    "Ratio:",
    max(counts) / min(counts)
)


Most frequent class: person 5019
Least frequent class: bus 317
Ratio: 15.832807570977918


In [46]:
from collections import defaultdict
import numpy as np
import tensorflow as tf

train_dataset = build_yolo_dataset(
    batch_size=16,
    split="train",
    shuffle=False
)

class_obj_scores = defaultdict(list)

for batch in train_dataset:
    images = batch["image"]
    targets = batch["targets"]

    preds = model(images, training=False)

    object_mask = targets[..., 0] > 0

    for class_id in range(20):
        class_mask = (
            object_mask &
            (tf.argmax(targets[..., 5:], axis=-1) == class_id)
        )

        scores = tf.sigmoid(preds[..., 0])
        class_scores = tf.boolean_mask(scores, class_mask)

        if tf.size(class_scores) > 0:
            class_obj_scores[class_id].extend(
                class_scores.numpy().tolist()
            )

print("Average positive objectness by class:\n")

for class_id in range(20):
    scores = class_obj_scores[class_id]

    if scores:
        print(
            f"{class_id:2d} "
            f"{ID_TO_CLASS[class_id]:12s} "
            f"n={len(scores):4d} "
            f"avg_obj={np.mean(scores):.4f}"
        )

Average positive objectness by class:

 0 aeroplane    n= 470 avg_obj=0.1383
 1 bicycle      n= 410 avg_obj=0.0702
 2 bird         n= 592 avg_obj=0.0808
 3 boat         n= 508 avg_obj=0.0746
 4 bottle       n= 744 avg_obj=0.0484
 5 bus          n= 317 avg_obj=0.0920
 6 car          n=1191 avg_obj=0.0743
 7 cat          n= 609 avg_obj=0.0986
 8 chair        n=1457 avg_obj=0.0538
 9 cow          n= 355 avg_obj=0.1380
10 diningtable  n= 373 avg_obj=0.0571
11 dog          n= 768 avg_obj=0.0931
12 horse        n= 377 avg_obj=0.1264
13 motorbike    n= 375 avg_obj=0.1096
14 person       n=5010 avg_obj=0.0857
15 pottedplant  n= 557 avg_obj=0.0409
16 sheep        n= 508 avg_obj=0.1627
17 sofa         n= 399 avg_obj=0.0664
18 train        n= 327 avg_obj=0.0716
19 tvmonitor    n= 412 avg_obj=0.0433


In [47]:
import numpy as np
import tensorflow as tf
from src.dataset import build_yolo_dataset

train_dataset = build_yolo_dataset(
    batch_size=16,
    split="train",
    shuffle=False
)

positive_counts = []
positive_bce_means = []
positive_objectness_means = []

for batch in train_dataset:

    images = batch["image"]
    targets = batch["targets"]

    preds = model(images, training=False)

    true_obj = targets[..., 0]
    pred_obj_logits = preds[..., 0]

    object_mask = true_obj > 0

    # Positive objectness probability
    pred_obj_probs = tf.sigmoid(pred_obj_logits)

    # BCE for each prediction
    bce = tf.nn.sigmoid_cross_entropy_with_logits(
        labels=true_obj,
        logits=pred_obj_logits
    )

    batch_size = images.shape[0]

    for i in range(batch_size):

        # Positive locations for this image
        mask = object_mask[i]

        count = int(tf.reduce_sum(tf.cast(mask, tf.int32)))

        if count == 0:
            positive_counts.append(0)
            positive_bce_means.append(0.0)
            positive_objectness_means.append(0.0)
            continue

        image_positive_bce = tf.boolean_mask(
            bce[i],
            mask
        )

        image_positive_obj = tf.boolean_mask(
            pred_obj_probs[i],
            mask
        )

        positive_counts.append(count)

        positive_bce_means.append(
            float(tf.reduce_mean(image_positive_bce))
        )

        positive_objectness_means.append(
            float(tf.reduce_mean(image_positive_obj))
        )

print("Number of training images:", len(positive_counts))

Number of training images: 5717


In [48]:
print("\nPositive slots per image:")
print(
    "min =", min(positive_counts),
    "max =", max(positive_counts),
    "mean =", np.mean(positive_counts)
)

print("\nMean positive BCE per image:")
print(
    "min =", np.min(positive_bce_means),
    "max =", np.max(positive_bce_means),
    "mean =", np.mean(positive_bce_means),
    "median =", np.median(positive_bce_means)
)

print("\nMean positive objectness per image:")
print(
    "min =", np.min(positive_objectness_means),
    "max =", np.max(positive_objectness_means),
    "mean =", np.mean(positive_objectness_means),
    "median =", np.median(positive_objectness_means)
)


Positive slots per image:
min = 1 max = 55 mean = 2.7565156550638448

Mean positive BCE per image:
min = 0.2992044985294342 max = 8.946040153503418 mean = 2.96736830906662 median = 2.986605405807495

Mean positive objectness per image:
min = 0.00013025198131799698 max = 0.7414076924324036 mean = 0.09319131703422912 median = 0.06176001578569412


In [49]:
print("\nPositive objectness percentiles:")
for p in [10, 25, 50, 75, 90]:
    print(
        f"{p}th percentile:",
        np.percentile(positive_objectness_means, p)
    )

print("\nPositive BCE percentiles:")
for p in [10, 25, 50, 75, 90]:
    print(
        f"{p}th percentile:",
        np.percentile(positive_bce_means, p)
    )


Positive objectness percentiles:
10th percentile: 0.01841723024845123
25th percentile: 0.033876944333314896
50th percentile: 0.06176001578569412
75th percentile: 0.11485689133405685
90th percentile: 0.2120869278907776

Positive BCE percentiles:
10th percentile: 1.6146518468856812
25th percentile: 2.3071656227111816
50th percentile: 2.986605405807495
75th percentile: 3.5838913917541504
90th percentile: 4.194201278686523


In [50]:
print(
    "Images with zero positive slots:",
    sum(c == 0 for c in positive_counts)
)

Images with zero positive slots: 0
